In [3]:
from config import Config
from core.mesh import Mesh  # (Assume this handles area/volume calcs)
from core.fvm_solver import FVMSolver
from core.kinetics import KineticsEngine
from utils.io_manager import IOManager
import numpy as np

def run_simulation():
    # 1. Setup
    cfg = Config()
    mesh = Mesh(cfg)
    solver = FVMSolver(cfg, mesh)
    kinetics = KineticsEngine(cfg)
    io = IOManager(cfg)

    # 2. Initialize State
    T = np.full((cfg.NR, cfg.NZ), cfg.T_INITIAL)
    W = {c: np.full((cfg.NR, cfg.NZ), cfg.W0[c]) for c in cfg.W0}

    # 3. Time Loop
    current_sec = 0
    while current_sec <= (cfg.TOTAL_MIN * 60):
        t_min = current_sec / 60.0
        T_amb = cfg.T_INITIAL + (cfg.HEATING_RATE * t_min)

        # Physics steps
        T = solver.solve_step(T, T_amb, cfg.DT)
        W = kinetics.update_mass(W, T, cfg.DT)

        # Saving / Logging
        if int(current_sec) % 60 == 0:
            io.save_data(T, W, current_sec)
            print(f"Time: {t_min:.1f} min | Avg T: {np.mean(T):.1f}")

        current_sec += cfg.DT

    io.finalize()

if __name__ == "__main__":
    run_simulation()

ModuleNotFoundError: No module named 'core'

In [2]:
pip install Config